# Phase 2 — SQL Analytical Queries

This notebook creates and tests the analytical DuckDB query layer used by later visualization, regression, Uri, duck-curve, and forecast notebooks.

In [1]:

import duckdb
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

conn = duckdb.connect(str(PROJECT_ROOT / 'ercot_intelligence.db'))

print("Connected to DuckDB: OK")
print(f"Project root: {PROJECT_ROOT}")

tables = conn.execute("""
    SELECT table_name
    FROM information_schema.tables
    WHERE table_schema = 'main'
    ORDER BY table_name
""").df()
print("Available tables:")
print(tables.to_string(index=False))

for table in tables['table_name']:
    count = conn.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
    print(f"  {table}: {count:,} rows")


Connected to DuckDB: OK
Project root: /Users/chokkapusaatvika/Desktop/ERCO_Project
Available tables:
                   table_name
          duck_curve_analysis
             ercot_generation
                   ercot_load
                   gas_prices
       monthly_generation_mix
price_cannibalization_dataset
      renewable_share_monthly
           storm_uri_analysis
               storm_uri_load
  duck_curve_analysis: 720 rows
  ercot_generation: 840 rows
  ercot_load: 789,039 rows
  gas_prices: 120 rows
  monthly_generation_mix: 840 rows
  price_cannibalization_dataset: 120 rows
  renewable_share_monthly: 120 rows
  storm_uri_analysis: 264 rows
  storm_uri_load: 6,048 rows


In [2]:

# Execute generation mix SQL. The script creates reusable views and returns monthly_generation_mix.
gen_mix_sql = (PROJECT_ROOT / 'sql/02_generation_mix_queries.sql').read_text()
gen_mix = conn.execute(gen_mix_sql).df()
renewable_df = conn.execute('SELECT * FROM renewable_share_monthly ORDER BY year, month').df()

print(f"Generation mix result: {gen_mix.shape}")
print(gen_mix.head(20).to_string(index=False))

print("\nRenewable share result:")
print(renewable_df.head(20).to_string(index=False))

print("\nFuel type shares in 2015 vs 2024:")
for fuel in ['wind', 'solar', 'natural_gas', 'coal', 'nuclear']:
    share_2015 = gen_mix[(gen_mix['year'] == 2015) & (gen_mix['fuel_type'] == fuel)]['pct_share'].mean()
    share_2024 = gen_mix[(gen_mix['year'] == 2024) & (gen_mix['fuel_type'] == fuel)]['pct_share'].mean()
    print(f"  {fuel}: {share_2015:.1f}% (2015) -> {share_2024:.1f}% (2024)")


Generation mix result: (840, 10)
 year  month month_date   fuel_type  generation_mwh  total_generation_mwh  pct_share  pct_share_prior_year  yoy_change_ppts  rolling_12m_avg_pct
 2015      1 2015-01-01        coal      11365000.0            38164000.0     29.779                   NaN              NaN            29.779000
 2015      1 2015-01-01       hydro        131000.0            38164000.0      0.343                   NaN              NaN             0.343000
 2015      1 2015-01-01 natural_gas      19252000.0            38164000.0     50.445                   NaN              NaN            50.445000
 2015      1 2015-01-01     nuclear       3817000.0            38164000.0     10.002                   NaN              NaN            10.002000
 2015      1 2015-01-01       other        536000.0            38164000.0      1.404                   NaN              NaN             1.404000
 2015      1 2015-01-01       solar         32000.0            38164000.0      0.084             

In [3]:

# Execute price cannibalization dataset SQL.
price_sql = (PROJECT_ROOT / 'sql/03_price_analysis_queries.sql').read_text()
price_df = conn.execute(price_sql).df()

print(f"Price regression dataset: {price_df.shape}")
print(price_df.head(20).to_string(index=False))
print("\nDescribe:")
print(price_df[['renewable_pct', 'solar_pct', 'wind_pct', 'henry_hub_price', 'implied_wholesale_price_proxy']].describe().to_string())
print("\nCorrelation matrix:")
print(price_df[['renewable_pct', 'henry_hub_price', 'implied_wholesale_price_proxy']].corr().round(3).to_string())


Price regression dataset: (120, 20)
 year  month month_date  total_generation_mwh  renewable_mwh  wind_mwh  solar_mwh    gas_mwh   coal_mwh  renewable_pct  solar_pct  wind_pct  renewable_pct_prior_year  renewable_yoy_change   season  peak_season_flag  henry_hub_price  henry_hub_max  henry_hub_min  implied_wholesale_price_proxy
 2015      1 2015-01-01            38164000.0      3063000.0 3031000.0    32000.0 19252000.0 11365000.0          8.026      0.084     7.942                       NaN                   NaN   winter                 1         2.994500           3.32           2.88                      52.403750
 2015      2 2015-02-01            33449000.0      3301000.0 3268000.0    33000.0 16980000.0  9146000.0          9.869      0.099     9.770                       NaN                   NaN   winter                 1         2.874737           3.22           2.62                      50.307895
 2015      3 2015-03-01            33042000.0      2586000.0 2544000.0    42000.0 188

In [4]:

# Execute Winter Storm Uri analysis SQL.
uri_sql = (PROJECT_ROOT / 'sql/04_storm_uri_queries.sql').read_text()
uri_df = conn.execute(uri_sql).df()

print(f"Uri analysis rows: {len(uri_df)}")
print(uri_df.head(20).to_string(index=False))
print(f"\nPeak demand during storm window: {uri_df['demand_mw'].max():,.0f} MW")
print(f"Average baseline Feb demand: {uri_df['baseline_load_mw'].mean():,.0f} MW")
print(f"Peak demand surge: {uri_df['demand_surge_mw'].max():,.0f} MW above baseline")
print(f"Hours with outage flag: {uri_df['outage_period_flag'].sum()}")


Uri analysis rows: 264
           datetime  year  month  day  hour    demand_mw  baseline_load_mw  demand_surge_mw  outage_period_flag  peak_demand_in_period  min_demand_in_period  avg_baseline
2021-02-10 00:00:00  2021      2   10     0 47723.784739      37135.696854     10588.087885                   0           69692.485103           33953.36042  39539.965559
2021-02-10 01:00:00  2021      2   10     1 40520.626171      35786.848655      4733.777516                   0           69692.485103           33953.36042  39539.965559
2021-02-10 02:00:00  2021      2   10     2 39990.922868      35100.176278      4890.746590                   0           69692.485103           33953.36042  39539.965559
2021-02-10 03:00:00  2021      2   10     3 39947.716182      34877.255225      5070.460957                   0           69692.485103           33953.36042  39539.965559
2021-02-10 04:00:00  2021      2   10     4 40278.917793      35047.521300      5231.396493                   0           

In [5]:

# Execute duck-curve query SQL.
duck_sql = (PROJECT_ROOT / 'sql/05_duck_curve_queries.sql').read_text()
duck_df = conn.execute(duck_sql).df()

print(f"Duck curve dataset: {duck_df.shape}")
print(duck_df.head(30).to_string(index=False))
print("\nDuck curve depth by year (summer):")
depth_by_year = duck_df[duck_df['season'] == 'summer'].groupby('year')['duck_curve_depth_mw'].first()
print(depth_by_year.to_string())


Duck curve dataset: (720, 10)
 year  hour   season  avg_load_mw  avg_wind_mw  avg_solar_mw  net_load_mw  season_min_net_load  evening_peak_net_load  duck_curve_depth_mw
 2015     0 shoulder 34109.379963  5213.157109     66.509857 28829.712998         23642.758510           37620.215376         13977.456866
 2015     1 shoulder 31798.420263  5213.157109     66.509857 26518.753297         23642.758510           37620.215376         13977.456866
 2015     2 shoulder 30293.043464  5213.157109     66.509857 25013.376498         23642.758510           37620.215376         13977.456866
 2015     3 shoulder 29371.643789  5213.157109     66.509857 24091.976824         23642.758510           37620.215376         13977.456866
 2015     4 shoulder 28922.425475  5213.157109     66.509857 23642.758510         23642.758510           37620.215376         13977.456866
 2015     5 shoulder 29102.400587  5213.157109     66.509857 23822.733622         23642.758510           37620.215376         13977.4568

In [6]:

# Export all query results for downstream notebooks and Tableau.
processed_dir = PROJECT_ROOT / 'data/processed'
tableau_dir = PROJECT_ROOT / 'outputs/tableau_exports'
processed_dir.mkdir(parents=True, exist_ok=True)
tableau_dir.mkdir(parents=True, exist_ok=True)

gen_mix.to_csv(processed_dir / 'gen_mix_analysis.csv', index=False)
renewable_df.to_csv(processed_dir / 'renewable_share.csv', index=False)
price_df.to_csv(processed_dir / 'price_analysis.csv', index=False)
uri_df.to_csv(processed_dir / 'uri_analysis.csv', index=False)
duck_df.to_csv(processed_dir / 'duck_curve.csv', index=False)

print("All analytical DataFrames saved to data/processed/")
print("Sizes:")
for name, df in [('gen_mix', gen_mix), ('renewable', renewable_df), ('price', price_df), ('uri', uri_df), ('duck', duck_df)]:
    print(f"  {name}: {df.shape}")

gen_mix.to_csv(tableau_dir / 'monthly_generation_mix.csv', index=False)
price_df.to_csv(tableau_dir / 'price_analysis.csv', index=False)
uri_df.to_csv(tableau_dir / 'storm_uri_hourly.csv', index=False)
duck_df.to_csv(tableau_dir / 'duck_curve.csv', index=False)
renewable_df.to_csv(tableau_dir / 'renewable_share.csv', index=False)
print("Tableau export CSVs saved")
for path in sorted(tableau_dir.glob('*.csv')):
    print(f"  {path.relative_to(PROJECT_ROOT)}: {len(pd.read_csv(path)):,} rows")


All analytical DataFrames saved to data/processed/
Sizes:
  gen_mix: (840, 10)
  renewable: (120, 14)
  price: (120, 20)
  uri: (264, 12)
  duck: (720, 10)
Tableau export CSVs saved
  outputs/tableau_exports/duck_curve.csv: 720 rows
  outputs/tableau_exports/monthly_generation_mix.csv: 840 rows
  outputs/tableau_exports/price_analysis.csv: 120 rows
  outputs/tableau_exports/renewable_forecast.csv: 72 rows
  outputs/tableau_exports/renewable_share.csv: 120 rows
  outputs/tableau_exports/revenue_adequacy_index.csv: 10 rows
  outputs/tableau_exports/scarcity_hours.csv: 10 rows
  outputs/tableau_exports/storm_uri_hourly.csv: 264 rows


In [7]:

print("=" * 60)
print("PHASE 2 GATE CHECK")
print("=" * 60)

checks = {}

for name, df in [('gen_mix', gen_mix), ('renewable_df', renewable_df), ('price_df', price_df), ('uri_df', uri_df), ('duck_df', duck_df)]:
    checks[f'{name}_has_data'] = len(df) > 0 and not df.empty

checks['gen_mix_year_coverage'] = gen_mix['year'].nunique() == 10
checks['gen_mix_fuel_coverage'] = gen_mix['fuel_type'].nunique() == 7

renewable_2015 = renewable_df[renewable_df['year'] == 2015]['renewable_pct'].mean()
renewable_2024 = renewable_df[renewable_df['year'] == 2024]['renewable_pct'].mean()
checks['renewable_share_grew'] = renewable_2024 > renewable_2015
print(f"  Renewable share 2015: {renewable_2015:.1f}%")
print(f"  Renewable share 2024: {renewable_2024:.1f}%")

checks['price_df_has_gas'] = price_df['henry_hub_price'].notna().sum() > 50

uri_peak = uri_df['demand_mw'].max()
checks['uri_shows_demand_spike'] = uri_peak > 60000
print(f"  Uri peak demand: {uri_peak:,.0f} MW")

duck_2019 = duck_df[(duck_df['year'] == 2019) & (duck_df['season'] == 'summer')]['duck_curve_depth_mw'].max()
duck_2024 = duck_df[(duck_df['year'] == 2024) & (duck_df['season'] == 'summer')]['duck_curve_depth_mw'].max()
checks['duck_curve_deepening'] = duck_2024 > duck_2019
print(f"  Duck curve depth 2019: {duck_2019:,.0f} MW")
print(f"  Duck curve depth 2024: {duck_2024:,.0f} MW")

tableau_files = list((PROJECT_ROOT / 'outputs/tableau_exports').glob('*.csv'))
checks['tableau_exports_saved'] = len(tableau_files) >= 4
print(f"  Tableau export files: {len(tableau_files)}")

sql_files = list((PROJECT_ROOT / 'sql').glob('*.sql'))
checks['sql_files_saved'] = len(sql_files) >= 5
print(f"  SQL files: {len(sql_files)}")

print("\nGate check results:")
all_passed = True
for check_name, passed in checks.items():
    status = "PASS" if passed else "FAIL"
    print(f"  [{status}] {check_name}")
    if not passed:
        all_passed = False

print("\n" + "=" * 60)
if all_passed:
    print("ALL PHASE 2 CHECKS PASSED")
    print("\nKey findings so far:")
    print(f"  Renewable share 2015 -> 2024: {renewable_2015:.1f}% -> {renewable_2024:.1f}%")
    print(f"  Uri peak demand: {uri_peak:,.0f} MW")
    print(f"  Duck curve deepened: {duck_2019:,.0f} MW -> {duck_2024:,.0f} MW")
else:
    print("PHASE 2 HAS FAILING CHECKS — DO NOT PROCEED")
    for check_name, passed in checks.items():
        if not passed:
            print(f"  FAIL: {check_name}")
print("=" * 60)

print("""
Phase 2 gate check complete.
Please review:
- All 5 analytical DataFrames printed above
- Key statistics: renewable share growth, Uri peak demand, duck curve depth
- Tableau export files in outputs/tableau_exports/
- SQL files in sql/ directory

Type APPROVED to proceed to Phase 3 (statistical analysis and charts),
or describe any issues to fix.
""")

conn.close()


PHASE 2 GATE CHECK
  Renewable share 2015: 10.3%
  Renewable share 2024: 30.2%
  Uri peak demand: 69,692 MW
  Duck curve depth 2019: 24,746 MW
  Duck curve depth 2024: 25,268 MW
  Tableau export files: 8
  SQL files: 5

Gate check results:
  [PASS] gen_mix_has_data
  [PASS] renewable_df_has_data
  [PASS] price_df_has_data
  [PASS] uri_df_has_data
  [PASS] duck_df_has_data
  [PASS] gen_mix_year_coverage
  [PASS] gen_mix_fuel_coverage
  [PASS] renewable_share_grew
  [PASS] price_df_has_gas
  [PASS] uri_shows_demand_spike
  [PASS] duck_curve_deepening
  [PASS] tableau_exports_saved
  [PASS] sql_files_saved

ALL PHASE 2 CHECKS PASSED

Key findings so far:
  Renewable share 2015 -> 2024: 10.3% -> 30.2%
  Uri peak demand: 69,692 MW
  Duck curve deepened: 24,746 MW -> 25,268 MW

Phase 2 gate check complete.
Please review:
- All 5 analytical DataFrames printed above
- Key statistics: renewable share growth, Uri peak demand, duck curve depth
- Tableau export files in outputs/tableau_exports/
- 

## Phase 3 — Generation Mix Transformation Charts

The following cells calculate publication-ready generation mix statistics and save Plotly charts as HTML plus PNG screenshots. PNG export uses the approved headless Chrome workaround because Kaleido could not launch Chrome in this local Python 3.13/macOS environment.

In [8]:

import subprocess
import textwrap

CHROME_PATH = PROJECT_ROOT / '.kaleido_chrome/chrome-mac-arm64/Google Chrome for Testing.app/Contents/MacOS/Google Chrome for Testing'
CHART_DIR = PROJECT_ROOT / 'outputs/charts'
CHART_DIR.mkdir(parents=True, exist_ok=True)

COLOR_MAP = {
    'natural_gas': '#636363',
    'wind': '#2196F3',
    'solar': '#FFC107',
    'coal': '#212121',
    'nuclear': '#9C27B0',
    'hydro': '#00BCD4',
    'other': '#BDBDBD',
}

FUEL_ORDER = ['coal', 'nuclear', 'hydro', 'other', 'natural_gas', 'wind', 'solar']


def apply_chart_standard(fig, title, xaxis_title, yaxis_title, source='Source: U.S. Energy Information Administration'):
    fig.update_layout(
        title=dict(text=title, font=dict(size=18, family='Arial', color='black')),
        font=dict(family='Arial', size=12),
        xaxis_title=xaxis_title,
        yaxis_title=yaxis_title,
        plot_bgcolor='white',
        paper_bgcolor='white',
        hovermode='x unified',
        legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='left', x=0),
        margin=dict(l=70, r=40, t=90, b=80),
    )
    fig.update_xaxes(showgrid=True, gridcolor='#F5F5F5', title_font=dict(size=14))
    fig.update_yaxes(showgrid=True, gridcolor='#F5F5F5', title_font=dict(size=14))
    fig.add_annotation(
        xref='paper', yref='paper', x=1.0, y=-0.14,
        text=source,
        showarrow=False, font=dict(size=10, color='gray'), xanchor='right'
    )
    return fig


def save_chart(fig, filename):
    html_path = CHART_DIR / f'{filename}.html'
    png_path = CHART_DIR / f'{filename}.png'
    fig.write_html(html_path)
    print(f'Saved: {html_path.relative_to(PROJECT_ROOT)}')

    if not CHROME_PATH.exists():
        raise FileNotFoundError(f'Chrome-for-Testing not found at {CHROME_PATH}')

    subprocess.run([
        str(CHROME_PATH),
        '--headless=new',
        '--disable-gpu',
        '--no-sandbox',
        '--disable-dev-shm-usage',
        f'--screenshot={png_path}',
        '--window-size=1200,700',
        f'file://{html_path}',
    ], check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    print(f'Saved: {png_path.relative_to(PROJECT_ROOT)}')

print('Chart export method: Plotly HTML + approved headless Chrome PNG screenshot')
print(f'Chrome path exists: {CHROME_PATH.exists()}')


Chart export method: Plotly HTML + approved headless Chrome PNG screenshot
Chrome path exists: True


In [9]:

print('GENERATION MIX TRANSFORMATION STATISTICS')
print('=' * 50)

for fuel in ['wind', 'solar', 'natural_gas', 'coal', 'nuclear']:
    avg_2015 = gen_mix[(gen_mix['year'] == 2015) & (gen_mix['fuel_type'] == fuel)]['pct_share'].mean()
    avg_2024 = gen_mix[(gen_mix['year'] == 2024) & (gen_mix['fuel_type'] == fuel)]['pct_share'].mean()
    total_2015 = gen_mix[(gen_mix['year'] == 2015) & (gen_mix['fuel_type'] == fuel)]['generation_mwh'].sum() / 1e6
    total_2024 = gen_mix[(gen_mix['year'] == 2024) & (gen_mix['fuel_type'] == fuel)]['generation_mwh'].sum() / 1e6
    print(f"\n{fuel.upper()}:")
    print(f"  Share: {avg_2015:.1f}% (2015) -> {avg_2024:.1f}% (2024)")
    print(f"  Total: {total_2015:.1f} TWh (2015) -> {total_2024:.1f} TWh (2024)")
    print(f"  Change: {avg_2024 - avg_2015:+.1f} percentage points")

solar_2019 = gen_mix[(gen_mix['year'] == 2019) & (gen_mix['fuel_type'] == 'solar')]['generation_mwh'].sum()
solar_2024 = gen_mix[(gen_mix['year'] == 2024) & (gen_mix['fuel_type'] == 'solar')]['generation_mwh'].sum()
print(f"\nSolar growth 2019 -> 2024: {(solar_2024 / solar_2019 - 1) * 100:.0f}%")

max_renewable = renewable_df.loc[renewable_df['renewable_pct'].idxmax()]
print(f"\nRecord renewable share month: {int(max_renewable['year'])}-{int(max_renewable['month']):02d}: {max_renewable['renewable_pct']:.1f}%")

renewable_2015 = renewable_df[renewable_df['year'] == 2015]['renewable_pct'].mean()
renewable_2024 = renewable_df[renewable_df['year'] == 2024]['renewable_pct'].mean()
print(f"\nAnnual average renewable share: {renewable_2015:.1f}% (2015) -> {renewable_2024:.1f}% (2024)")


GENERATION MIX TRANSFORMATION STATISTICS

WIND:
  Share: 10.2% (2015) -> 22.3% (2024)
  Total: 44.8 TWh (2015) -> 124.3 TWh (2024)
  Change: +12.2 percentage points

SOLAR:
  Share: 0.1% (2015) -> 7.9% (2024)
  Total: 0.6 TWh (2015) -> 45.5 TWh (2024)
  Change: +7.7 percentage points

NATURAL_GAS:
  Share: 52.6% (2015) -> 50.7% (2024)
  Total: 237.7 TWh (2015) -> 293.2 TWh (2024)
  Change: -1.9 percentage points

COAL:
  Share: 26.6% (2015) -> 11.3% (2024)
  Total: 121.6 TWh (2015) -> 65.5 TWh (2024)
  Change: -15.3 percentage points

NUCLEAR:
  Share: 8.8% (2015) -> 6.8% (2024)
  Total: 39.4 TWh (2015) -> 38.6 TWh (2024)
  Change: -1.9 percentage points

Solar growth 2019 -> 2024: 748%

Record renewable share month: 2024-04: 38.7%

Annual average renewable share: 10.3% (2015) -> 30.2% (2024)


In [10]:

fig = px.area(
    gen_mix.sort_values(['year', 'month', 'fuel_type']),
    x='month_date',
    y='generation_mwh',
    color='fuel_type',
    color_discrete_map=COLOR_MAP,
    category_orders={'fuel_type': FUEL_ORDER},
    labels={
        'generation_mwh': 'Generation (MWh)',
        'month_date': 'Date',
        'fuel_type': 'Fuel Type',
    },
)

fig = apply_chart_standard(
    fig,
    'ERCOT Generation Mix Transformation 2015-2024',
    'Date',
    'Generation (MWh)',
)

fig.add_annotation(
    x='2023-06-01',
    y=gen_mix['generation_mwh'].max() * 0.90,
    text=f"Solar generation<br>grew {(solar_2024 / solar_2019 - 1) * 100:.0f}% since 2019",
    showarrow=True,
    arrowhead=2,
    bgcolor='white',
    bordercolor='black',
    font=dict(size=11),
)

save_chart(fig, 'chart_1a_generation_mix_stacked')
fig.show()


Saved: outputs/charts/chart_1a_generation_mix_stacked.html


Saved: outputs/charts/chart_1a_generation_mix_stacked.png


In [11]:

x_numeric = (renewable_df['year'] - 2015) * 12 + renewable_df['month']
trend_coeffs = np.polyfit(x_numeric, renewable_df['renewable_pct'], 1)
trend_line = np.poly1d(trend_coeffs)(x_numeric)
trend_ppts_per_year = trend_coeffs[0] * 12

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=renewable_df['month_date'],
    y=renewable_df['renewable_pct'],
    mode='lines',
    name='Renewable Share',
    line=dict(color='#4CAF50', width=2),
))
fig.add_trace(go.Scatter(
    x=renewable_df['month_date'],
    y=trend_line,
    mode='lines',
    name=f'Trend (+{trend_ppts_per_year:.1f} ppts/year)',
    line=dict(color='black', width=1.5, dash='dash'),
))

for level, label in [(20, '20%'), (30, '30%'), (40, '40%')]:
    fig.add_hline(y=level, line_dash='dot', line_color='lightgray', annotation_text=label, annotation_position='right')

uri_marker = pd.Timestamp('2021-02-15')
fig.add_shape(
    type='line', x0=uri_marker, x1=uri_marker, y0=0, y1=1,
    xref='x', yref='paper', line=dict(color='red', dash='dash')
)
fig.add_annotation(
    x=uri_marker, y=1.06, xref='x', yref='paper',
    text='Winter Storm Uri', showarrow=False, font=dict(size=11, color='red')
)

fig = apply_chart_standard(
    fig,
    'ERCOT Renewable Share Trend 2015-2024',
    'Date',
    'Renewable Share (% of total generation)',
)
fig.update_yaxes(ticksuffix='%')

print(f'Renewable share trend: +{trend_ppts_per_year:.2f} percentage points per year')
save_chart(fig, 'chart_1b_renewable_share_trend')
fig.show()


Renewable share trend: +2.16 percentage points per year
Saved: outputs/charts/chart_1b_renewable_share_trend.html


Saved: outputs/charts/chart_1b_renewable_share_trend.png
